<a href="https://colab.research.google.com/github/internshipinabook/python-internship-in-a-book/blob/main/notebooks/week7_pandas_II_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/python-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 7 -- pandas II: Cleaning, Grouping, and Transforming
### The Python Internship | Meridian Consulting

> **STARTER notebook.** Work Monday to Friday in order.
> Dataset: `golden_fork_orders.csv` (520 orders, 13 columns)

---

<a href="https://colab.research.google.com/github/internshipinabook/python-internship-in-a-book/blob/main/notebooks/week7/week7_pandas_II_STARTER.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>&nbsp;&nbsp;
<a href="https://github.com/internshipinabook/python-internship-in-a-book/blob/main/notebooks/week7/week7_pandas_II_STARTER.ipynb">
  <img src="https://img.shields.io/badge/View%20on-GitHub-181717?logo=github" alt="View on GitHub"/>
</a>

---

## 🖥️ How to Run This Notebook

| Platform | Instructions |
|----------|-------------|
| **Google Colab** | Click the badge above — free, no setup needed |
| **Local Jupyter** | `pip install -r requirements.txt` then `jupyter lab` |
| **VS Code** | Open `.ipynb` with the Jupyter extension installed |
| **GitHub** | Click "View on GitHub" above for a read-only preview |

> **STARTER notebook** — Complete the `# YOUR CODE HERE` cells. Check the SOLUTION notebook if stuck.

In [ ]:
# ── PLATFORM SETUP ───────────────────────────────────────────────────────────
# Run this cell first. It installs missing libraries in Google Colab.
# In a local environment, skip it if requirements.txt is already installed.

import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("📦 Google Colab detected — installing libraries...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pandas>=1.5', 'numpy>=1.23', 'matplotlib>=3.6',
                    'seaborn>=0.12', 'scikit-learn>=1.2'], check=True)
    print("✅ Libraries installed.")
else:
    print("💻 Running locally.")
    print("   If you see ImportError below, run: pip install -r requirements.txt")

---

---

## This week covers

`isnull()` | `dropna()` | `fillna()` | `pd.to_datetime()` | `.astype()` | `pd.to_numeric()` | `.dt` accessor | `duplicated()` | `drop_duplicates()` | `.str` accessor | `df.rename()` | `groupby()` | `.agg()` | `reset_index()` | `.apply()` | `.map()` | `to_csv(index=False)` | cleaning log

---

## MONDAY -- Missing Values: Detect, Decide, Handle

Three questions first: What does absence mean? How much is missing? Does this column affect the analysis?

---

### Example 1 -- Detect

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv")

print("Missing counts:")
print(df.isnull().sum())
print("\nMissing proportions:")
print(df.isnull().mean().round(3))
# table_number: 105 missing values = 20.2% of rows

null_rows = df[df["table_number"].isnull()]
print(f"Rows with missing: {len(null_rows)}")
print(null_rows[["order_id","item_name","payment_method"]].head())

### Example 2 -- dropna() (use cautiously)

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv")
print(f"dropna() all: {len(df.dropna())} rows (was 520)")
print(f"dropna(subset): {len(df.dropna(subset=['table_number']))} rows")
# 105 missing = 20% of the dataset -- dropping loses too much.
# fillna() with a sentinel is the better choice.

### Example 3 -- fillna()

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv").copy()

# 0 means "takeaway / no table assigned"
df["table_number"] = df["table_number"].fillna(0)
print("Missing after fill:", df["table_number"].isnull().sum())  # 0

# Other strategies (reference):
# df["col"].fillna(df["col"].mean())    -- fill with mean
# df["col"].fillna(df["col"].median())  -- fill with median
# df["col"].ffill()                     -- forward fill

### Example 4 -- Cleaning log

In [ ]:
import pandas as pd
from datetime import datetime

cleaning_log = []

def log(column, action, before, after, reason):
    """Record one cleaning decision."""
    cleaning_log.append({
        "ts": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "column": column, "action": action,
        "before": str(before), "after": str(after), "reason": reason,
    })

df = pd.read_csv("golden_fork_orders.csv").copy()

before_n = df["table_number"].isnull().sum()   # 105
df["table_number"] = df["table_number"].fillna(0)
after_n  = df["table_number"].isnull().sum()   # 0

log("table_number", "fillna(0)",
    f"{before_n} missing", f"{after_n} missing",
    "Missing values assumed to be takeaway orders. 0 used as sentinel.")

for e in cleaning_log:
    print(f"[{e['ts']}] {e['column']} | {e['action']}")
    print(f"  {e['before']} -> {e['after']}")
    print(f"  {e['reason']}")

---
## MONDAY EXERCISES
---

### Exercise 7.1 -- Diagnose missing values

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv")

print(df.isnull().sum())
print(df.isnull().mean().round(3))

# Which column? How many? What proportion?
# ANSWER:

null_rows = df[df["table_number"].isnull()]
print(null_rows[["order_id","item_name","payment_method"]].head())
# Pattern in payment_method?
# ANSWER:

print("All-missing rows:", df.isnull().all(axis=1).sum())

# What does a missing table_number most likely mean?
# ANSWER:


### Exercise 7.2 -- Handle missing values with a log

In [ ]:
import pandas as pd
from datetime import datetime

df = pd.read_csv("golden_fork_orders.csv").copy()
cleaning_log = []

def log(column, action, before, after, reason):
    cleaning_log.append({"ts": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "column": column, "action": action,
        "before": str(before), "after": str(after), "reason": reason})

before_n = df["table_number"].isnull().sum()
df["table_number"] = df["table_number"].fillna(0)
after_n  = df["table_number"].isnull().sum()
log("table_number", "fillna(0)",
    f"{before_n} missing", f"{after_n} missing",
    "YOUR REASON HERE")

print("Missing after fill:", df["table_number"].isnull().sum())
for e in cleaning_log:
    print(f"[{e['ts']}] {e['column']} | {e['action']}")
    print(f"  {e['before']} -> {e['after']}")
    print(f"  {e['reason']}")

df.to_csv("golden_fork_clean_v1.csv", index=False)
print("Saved: golden_fork_clean_v1.csv")


---
## TUESDAY -- Data Types: Converting and Fixing

---

### Example 1 -- pd.to_datetime()

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv").copy()
print("Before:", df["date"].dtype)    # object

df["date"] = pd.to_datetime(df["date"])
print("After: ", df["date"].dtype)    # datetime64[ns]

df["month"] = df["date"].dt.month
df["week"]  = df["date"].dt.isocalendar().week.astype(int)
df["year"]  = df["date"].dt.year
print(df[["date","month","week","year"]].head(3))

jan = df[df["date"] >= "2024-01-01"]
mar = df[(df["date"] >= "2024-03-01") & (df["date"] < "2024-04-01")]
print(f"January: {len(jan)} | March: {len(mar)}")

### Example 2 -- astype() and category dtype

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv").copy()

# Fill NaN BEFORE astype(int) -- astype fails on NaN
df["table_number"] = df["table_number"].fillna(0).astype(int)
print(df["table_number"].dtype, df["table_number"].head().tolist())

df["category"]    = df["category"].astype("category")
df["time_of_day"] = df["time_of_day"].astype("category")
print(df["category"].cat.categories)

### Example 3 -- pd.to_numeric()

In [ ]:
import pandas as pd

messy = pd.Series(["1800","2200","N/A","1500","bad"])
clean = pd.to_numeric(messy, errors="coerce")
print(clean)
# "N/A" and "bad" become NaN instead of crashing

---
## TUESDAY EXERCISES
---

### Exercise 7.3 -- Convert the date column

In [ ]:
import pandas as pd
from datetime import datetime

df = pd.read_csv("golden_fork_orders.csv").copy()
cleaning_log = []

def log(column, action, before, after, reason):
    cleaning_log.append({"column": column, "action": action,
        "before": str(before), "after": str(after), "reason": reason})

before_dtype = str(df["date"].dtype)
df["date"] = ___________
log("date", "pd.to_datetime()", before_dtype, str(df["date"].dtype),
    "Enables month/week filtering.")

df["month"] = ___________
df["week"]  = ___________
df["year"]  = ___________
print(df[["date","month","week","year"]].head(3))

print(df["month"].value_counts().sort_index())
print(f"March revenue: N{df[df['month']==3]['total_price'].sum():,.2f}")


### Exercise 7.4 -- Type conversions

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv").copy()

# 1. table_number: fillna(0) then astype(int)
df["table_number"] = ___________
print("table_number:", df["table_number"].dtype)

# 2. category and time_of_day to "category" dtype
before_mem = df["category"].memory_usage(deep=True)
df["category"]    = ___________
df["time_of_day"] = ___________
print(f"category memory: {before_mem} -> {df['category'].memory_usage(deep=True)} bytes")

# 3. Verify dtypes converted in this exercise
#    (date conversion was covered in Exercise 7.3)
print(df[["table_number","category","time_of_day"]].dtypes)

# 4. Save
df.to_csv("golden_fork_clean_v2.csv", index=False)
print("Saved: golden_fork_clean_v2.csv")


---
## WEDNESDAY -- Duplicates and String Cleaning

---

### Example 1 -- Duplicates

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv")

print("Duplicate rows:     ", df.duplicated().sum())                     # 0
print("Duplicate order_ids:", df.duplicated(subset=["order_id"]).sum())  # 0
# No duplicates -- always check, never assume.

### Example 2 -- .str accessor

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv").copy()

# Strip whitespace
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df["item_name"] = df["item_name"].str.title()
df["server"]    = df["server"].str.title()

df["payment_lower"] = df["payment_method"].str.lower()
print(df["payment_lower"].unique())

# str.replace() for substring substitution
# regex=False: treat the pattern as a literal string, not a regex
df["payment_clean"] = df["payment_method"].str.replace("Transfer","Bank Transfer", regex=False)
print(df["payment_clean"].unique())

# Slicing for abbreviations
df["day_short"] = df["day_of_week"].str[:3]
print(df["day_short"].unique())

### Example 3 -- df.rename() with a copy

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv").copy()

# Use a new variable -- not df = df.rename() -- to avoid breaking later exercises
df_renamed = df.rename(columns={
    "order_id": "id", "item_name": "item",
    "payment_method": "payment", "discount_applied": "discounted",
})
print(df_renamed.columns.tolist())
# df itself is unchanged -- df["item_name"] still works

---
## WEDNESDAY EXERCISES
---

### Exercise 7.5 -- Check and clean

In [ ]:
import pandas as pd
from datetime import datetime

df = pd.read_csv("golden_fork_orders.csv").copy()

cleaning_log = []   # initialise before use

def log(column, action, before, after, reason):
    cleaning_log.append({"ts": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "column": column, "action": action,
        "before": str(before), "after": str(after), "reason": reason})

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate order_ids:", df.duplicated(subset=["order_id"]).sum())

# Strip whitespace from all object columns and log
for col in df.select_dtypes(include="object").columns:
    df[col] = ___________
log("all text cols","str.strip()","raw","stripped","Remove whitespace.")

# Normalise item_name and server
df["item_name"] = ___________
df["server"]    = ___________
log("item_name,server","str.title()","raw","title case","Standardise.")

# Add item_lower
df["item_lower"] = ___________
print(f"Title unique: {df['item_name'].nunique()} | Lower unique: {df['item_lower'].nunique()}")

# Print log
for e in cleaning_log:
    print(f"[{e['ts']}] {e['column']} | {e['action']}")


### Exercise 7.6 -- String operations

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv").copy()

print("Soup orders:", df["item_name"].str.contains("Soup",case=False).sum())

df["is_drink"]       = ___________   # True where category == "Drinks"
df["server_initial"] = ___________   # first letter of server name
df["short_day"]      = ___________   # first 3 chars of day_of_week

print(df["is_drink"].value_counts())
print(df["server_initial"].unique())
print(df["short_day"].unique())


---
## THURSDAY -- Groupby and Aggregation

`groupby()` = split → apply → combine.

---

### Example 1 -- Basic groupby

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv")

rev_by_cat = df.groupby("category")["total_price"].sum()
print(rev_by_cat.sort_values(ascending=False))

rev_df = df.groupby("category")["total_price"].sum().reset_index()
rev_df.columns = ["category","total_revenue"]
print(rev_df)

print(df.groupby("server")["total_price"].mean().round(2))
print(df.groupby("day_of_week")["order_id"].count())

### Example 2 -- .agg()

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv")

summary = df.groupby("category")["total_price"].agg(
    total="sum", mean="mean", count="count", maximum="max")
print(summary.round(2))

multi = df.groupby("category").agg(
    total_revenue=("total_price","sum"),
    order_count=("order_id","count"),
    avg_qty=("quantity","mean"),
)
print(multi.round(2))

### Example 3 -- Daily summary

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv")
df["date"] = pd.to_datetime(df["date"])

daily = df.groupby("date").agg(
    orders=("order_id","count"),
    total_revenue=("total_price","sum"),
    avg_order=("total_price","mean"),
    portions_sold=("quantity","sum"),
).round(2)

print(daily.head(5))
print(f"Best day: N{daily['total_revenue'].max():,.2f}")

---
## THURSDAY EXERCISES
---

### Exercise 7.7 -- Basic groupby

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv")

# 1. Total revenue per server -- which server handles the most total revenue?
print(df.groupby("server")["total_price"].sum().sort_values(ascending=False))

# 2. Average order value per time_of_day
print(df.groupby("time_of_day")["total_price"].mean().round(2))

# 3. Total portions per item (top 5)
print(df.groupby("item_name")["quantity"].sum().sort_values(ascending=False).head(5))

# 4. Orders per payment_method
print(df.groupby("payment_method")["order_id"].count())

# 5. Weekday vs weekend revenue
df["is_weekend"] = df["day_of_week"].isin(["Saturday","Sunday"])
print(df.groupby("is_weekend")["total_price"].agg(
    total="sum", count="count", mean="mean").round(2))


### Exercise 7.8 -- agg() and multi-level

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv")
df["date"]  = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month

# 1. Category summary sorted by revenue
cat_sum = df.groupby("category").agg(
    total_revenue=("total_price","sum"),
    order_count=("order_id","count"),
    avg_order=("total_price","mean"),
).round(2)
print(cat_sum.sort_values("total_revenue",ascending=False))

# 2. Server x category revenue
sc = df.groupby(["server","category"])["total_price"].sum().reset_index()
sc.columns = ["server","category","revenue"]
print(sc.sort_values("revenue",ascending=False).head(8))

# 3. Monthly revenue
monthly = df.groupby("month").agg(
    total_revenue=("total_price","sum"),
    order_count=("order_id","count")
).round(2)
print(monthly)
print(f"Best month: {monthly['total_revenue'].idxmax()}")

# 4. Daily summary (index=True: date is meaningful index)
daily = df.groupby("date").agg(
    orders=("order_id","count"),
    total_revenue=("total_price","sum"),
    avg_order=("total_price","mean"),
    portions_sold=("quantity","sum"),
).round(2)
daily.to_csv("golden_fork_daily_summary.csv", index=True)
print(f"Saved {len(daily)} daily rows")


---
## FRIDAY -- Adding Columns and the Cleaned Output

---

### Example 1 -- apply() and map()

In [ ]:
import pandas as pd

df = pd.read_csv("golden_fork_orders.csv").copy()

def get_tier(price):
    if price >= 8000:   return "Platinum"
    elif price >= 5000: return "Gold"
    elif price >= 2500: return "Silver"
    elif price >= 1000: return "Bronze"
    else:               return "Standard"

df["revenue_tier"] = df["total_price"].apply(get_tier)
print(df["revenue_tier"].value_counts())

day_map = {"Monday":"Mon","Tuesday":"Tue","Wednesday":"Wed",
           "Thursday":"Thu","Friday":"Fri","Saturday":"Sat","Sunday":"Sun"}
df["day_short"] = df["day_of_week"].map(day_map)
print(df[["day_of_week","day_short"]].drop_duplicates())

### The Friday Build -- Complete Cleaning Pipeline

In [ ]:
import pandas as pd
from datetime import datetime

df = pd.read_csv("golden_fork_orders.csv").copy()
cleaning_log = []
input_shape = df.shape   # capture before any changes

def log(col, action, before, after, reason):
    cleaning_log.append({"ts": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "column": col, "action": action,
        "before": str(before), "after": str(after), "reason": reason})

# Step 1: Missing values
bef = df["table_number"].isnull().sum()
df["table_number"] = df["table_number"].fillna(0).astype(int)
log("table_number","fillna(0)+astype(int)",f"{bef} missing (float)","0 missing (int)",
    "Missing = takeaway. Sentinel 0. Converted to int.")

# Step 2: Date
df["date"] = pd.to_datetime(df["date"])
log("date","pd.to_datetime()","object","datetime64[ns]","Enables date operations.")

# Step 3: Strings
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()
log("text cols","str.strip()","raw","stripped","Remove whitespace.")

# Step 4: Computed columns
df["month"]       = df["date"].dt.month
df["week"]        = df["date"].dt.isocalendar().week.astype(int)
df["is_weekend"]  = df["day_of_week"].isin(["Saturday","Sunday"])
df["list_price"]  = df["quantity"] * df["unit_price"]
df["discount_amt"] = df["list_price"] - df["total_price"]
df["revenue_tier"] = df["total_price"].apply(
    lambda x: "Platinum" if x>=8000 else "Gold" if x>=5000
              else "Silver" if x>=2500 else "Bronze" if x>=1000 else "Standard")
df["day_short"] = df["day_of_week"].map({"Monday":"Mon","Tuesday":"Tue","Wednesday":"Wed",
    "Thursday":"Thu","Friday":"Fri","Saturday":"Sat","Sunday":"Sun"})
log("new cols","computed","n/a",
    "month,week,is_weekend,list_price,discount_amt,revenue_tier,day_short",
    "Analytical columns for Week 8.")

# Step 5: Save
df.to_csv("golden_fork_clean_final.csv", index=False)
# Note: CSV does not store dtype info -- on reload use parse_dates=["date"]

# Step 6: Log file
with open("cleaning_log.txt","w") as f:
    f.write("THE GOLDEN FORK -- CLEANING LOG\n")
    f.write(f"Input:  {input_shape[0]} rows x {input_shape[1]} columns\n")
    f.write(f"Output: {len(df)} rows x {len(df.columns)} columns\n")
    f.write("=" * 60 + "\n\n")
    for i, e in enumerate(cleaning_log,1):
        f.write(f"Step {i}: {e['column']} | {e['action']}\n")
        f.write(f"  Before: {e['before']}  After: {e['after']}\n")
        f.write(f"  Reason: {e['reason']}\n\n")

raw_cols = set(pd.read_csv("golden_fork_orders.csv").columns)
print(f"Output: {len(df)} x {len(df.columns)} | Steps: {len(cleaning_log)}")
print(f"New cols: {[c for c in df.columns if c not in raw_cols]}")

### Extension challenges

In [ ]:
# Challenge A: Server performance table
# groupby server: total revenue, order count, avg order
# Save as server_performance.csv

# Challenge B: pd.cut() for revenue_tier
# bins   = [0, 1000, 2500, 5000, 8000, float("inf")]
# labels = ["Standard","Bronze","Silver","Gold","Platinum"]
# Does pd.cut() give same result as apply(get_tier)?

# Challenge C: transform() for weekly contribution
# df["weekly_rev"] = df.groupby("week")["total_price"].transform("sum")
# df["pct_of_week"] = df["total_price"] / df["weekly_rev"] * 100
# Which order is highest % of its week?


## ✅ What You Can Do After This Week

By the end of this week, you can:
- Merge two DataFrames on a shared key column — the equivalent of a SQL JOIN
- Reshape data between long and wide format using `melt()` and `pivot_table()`
- Sort, rank, and reindex data for presentation-ready output
- Apply custom transformation functions to a DataFrame with `.apply()`
- Produce a cross-tabulation summary that answers multi-dimensional business questions

*Add `week7_pandas_II.ipynb` to your internship portfolio.*

---

## Week 7 Checklist

- [ ] Mon 7.1-7.2: 105 missing found; fillna(0) applied; logged; saved v1
- [ ] Tue 7.3-7.4: date to datetime; month/week extracted; table_number to int; saved v2
- [ ] Wed 7.5-7.6: 0 duplicates confirmed; whitespace stripped; title-case applied
- [ ] Thu 7.7-7.8: groupby by server, time, item; agg() multi-stat; daily summary saved
- [ ] Fri: pipeline runs; clean_final.csv produced; cleaning_log.txt written
- [ ] One extension done
- [ ] Can explain: dropna vs fillna, index=False, groupby split-apply-combine, apply vs map, cleaning log purpose

---
## Week 7 Complete
Next: `week8/week8_*_STARTER.ipynb`

---
*The Python Internship | Meridian Consulting*